# v4 35k baseline 모델

Goal:

- Keep a fixed BSD10k split: train 80% / final test 20%.
- Use the same `dcase2026_task1_baseline` `BaseClassifier` and metrics.
- Add BSD35k v4-filtered data and evaluate on the untouched BSD10k final test set.

Datasets compared:

- `v4_ge2`, `v4_ge3`, `v4_ge4`: BSD10k train80 + BSD35k filtered by v4 pseudo confidence.


Important: v4 score is not a direct 1-5 classifier. This notebook follows the existing convention:

```text
predicted_confidence_score = 1 + 4 * v4_filter_score
ge2 = predicted_confidence_score >= 2
ge3 = predicted_confidence_score >= 3
ge4 = predicted_confidence_score >= 4
```

In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

HERE = Path.cwd()
if not (HERE / 'baseline_confidnce_train').exists():
    # Allows running this notebook from the notebooks/ directory.
    ROOT = HERE.parent
else:
    ROOT = HERE

sys.path.insert(0, str(ROOT / 'baseline_confidnce_train'))
sys.path.insert(0, str(ROOT / 'dcase2026_task1_baseline'))

import confidence_baseline_common as cbc

SEED = 1821
BASELINE_MODES = ('both',)
USE_KFOLD = True
N_FOLDS = 5
NUM_EPOCHS = 100
BATCH_SIZE = 64
OUTPUT_ROOT = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_baseline_model'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
DATASET_DIR = OUTPUT_ROOT / 'datasets'
for path in [OUTPUT_ROOT, PLOTS_DIR, DATASET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
cbc.seed_everything(SEED)

print('ROOT:', ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('device:', cbc.device_summary())

## 1. Fixed BSD10k 80/20 split

`final_test` is held out before any BSD35k addition and is used only for final evaluation.

In [ ]:
full_df, class_dict, top_class_dict = cbc.load_baseline_assets(ROOT)
train_pool, final_test, split_df = cbc.make_fixed_holdout(full_df, OUTPUT_ROOT, seed=SEED, test_size=0.2)

print('BSD10k full:', len(full_df))
print('BSD10k train_pool 80%:', len(train_pool))
print('BSD10k final_test 20%:', len(final_test))
print('classes in train_pool:', train_pool['class'].nunique())
print('classes in final_test:', final_test['class'].nunique())
display(split_df.head())

## 2. Load BSD35k v4 predictions and build DCASE-compatible rows

The existing v4 file already contains v2/v3-derived scores for BSD35k:

- `binary_mlp_prob` from v3 binary confidence model
- `fiveclass_score`, `fiveclass_p45` from v2 5-class model
- `v4_filter_score` from v4 rank-average ensemble

In [ ]:
BSD35K_V4_PATH = ROOT / 'outputs' / 'confidence_filter_v4' / 'predictions' / 'BSD35k-CS_filter_predictions_v4.csv'
assert BSD35K_V4_PATH.exists(), f'Missing file: {BSD35K_V4_PATH}'

def build_bsd35k_dcase_dataframe(v4_path: Path) -> pd.DataFrame:
    score_df = pd.read_csv(v4_path)
    score_df['sound_id'] = score_df['sound_id'].astype(str)
    top_col = 'class_top' if 'class_top' in score_df.columns else 'top_class'
    needed = ['sound_id', 'class', top_col, 'v4_filter_score', 'binary_mlp_prob', 'fiveclass_score', 'fiveclass_p45']
    missing = [col for col in needed if col not in score_df.columns]
    if missing:
        raise ValueError(f'Missing columns in BSD35k v4 file: {missing}')

    # Do NOT reuse score_df['class_idx']: in the BSD35k v4 file it is not the
    # DCASE baseline 0-22 class index. Re-map from class/top_class using
    # data/class_dict.json and data/top_class_dict.json.
    out = score_df[needed].copy().rename(columns={top_col: 'top_class'})
    out['index'] = out['sound_id'].astype(str)
    out['class'] = out['class'].astype(str)
    out['top_class'] = out['top_class'].astype(str)
    out['class_idx'] = out['class'].map(class_dict)
    out['top_class_idx'] = out['top_class'].map(top_class_dict)

    # BSD10k and BSD35k embeddings are stored in different folders.
    bsd35k_audio_dir = ROOT / 'data' / 'features' / 'BSD35k_clap_audio_embeddings'
    bsd35k_text_dir = ROOT / 'data' / 'features' / 'BSD35k-CS_clap_text_embeddings'
    if not bsd35k_audio_dir.exists():
        raise FileNotFoundError(f'Missing BSD35k audio embedding folder: {bsd35k_audio_dir}')
    if not bsd35k_text_dir.exists():
        raise FileNotFoundError(f'Missing BSD35k text embedding folder: {bsd35k_text_dir}')
    out['audio_emb_filepath'] = out['index'].map(lambda sid: str(bsd35k_audio_dir / f'{sid}.npy'))
    out['text_emb_filepath'] = out['index'].map(lambda sid: str(bsd35k_text_dir / f'{sid}.npy'))
    out['predicted_confidence_score'] = 1.0 + 4.0 * out['v4_filter_score'].astype(float)

    before = len(out)
    bsd10k_ids = set(full_df['index'].astype(str))
    out = out[~out['index'].isin(bsd10k_ids)].copy()

    unknown_class = out['class_idx'].isna().sum()
    unknown_top = out['top_class_idx'].isna().sum()
    missing_audio = (~out['audio_emb_filepath'].map(lambda p: Path(p).exists())).sum()
    missing_text = (~out['text_emb_filepath'].map(lambda p: Path(p).exists())).sum()
    print(f'Unknown DCASE class rows: {unknown_class:,}')
    print(f'Unknown DCASE top-class rows: {unknown_top:,}')
    print(f'Missing BSD35k audio embeddings: {missing_audio:,}')
    print(f'Missing BSD35k text embeddings: {missing_text:,}')

    out = out[out['class_idx'].notna() & out['top_class_idx'].notna()].copy()
    out = out[out['audio_emb_filepath'].map(lambda p: Path(p).exists())].copy()
    out = out[out['text_emb_filepath'].map(lambda p: Path(p).exists())].copy()
    out['class_idx'] = out['class_idx'].astype(int)
    out['top_class_idx'] = out['top_class_idx'].astype(int)
    out = out.reset_index(drop=True)
    print(f'BSD35k rows before compatibility filtering: {before:,}')
    print(f'BSD35k rows usable by DCASE baseline: {len(out):,}')
    print(f'BSD35k classes retained: {out["class"].nunique()}')
    return out

bsd35k_all = build_bsd35k_dcase_dataframe(BSD35K_V4_PATH)
bsd35k_all.to_csv(DATASET_DIR / 'bsd35k_all_usable.csv', index=False)
display(bsd35k_all.head())

## 3. Build v4 ge2/ge3/ge4 subsets and visualize counts

Each subset is created from BSD35k only. The BSD10k train80 anchor is added later during downstream training.

In [ ]:
THRESHOLD_SPECS = [
    {'label': 'ge2', 'threshold': 2.0},
    {'label': 'ge3', 'threshold': 3.0},
    {'label': 'ge4', 'threshold': 4.0},
]

v4_subsets = {}
for spec in THRESHOLD_SPECS:
    label = spec['label']
    threshold = spec['threshold']
    subset = bsd35k_all[bsd35k_all['predicted_confidence_score'] >= threshold].reset_index(drop=True)
    v4_subsets[label] = subset
    subset.to_csv(DATASET_DIR / f'bsd35k_v4_{label}.csv', index=False)

counts_rows = []
for label, subset in v4_subsets.items():
    counts_rows.append({
        'subset': label,
        'threshold': next(s['threshold'] for s in THRESHOLD_SPECS if s['label'] == label),
        'samples': len(subset),
        'ratio_of_usable_bsd35k': len(subset) / len(bsd35k_all),
        'num_classes': subset['class'].nunique(),
    })

subset_counts = pd.DataFrame(counts_rows)
subset_counts.to_csv(OUTPUT_ROOT / 'v4_bsd35k_subset_counts.csv', index=False)
display(subset_counts)

def plot_subset_total_and_class_counts(label: str, subset: pd.DataFrame):
    class_counts = subset['class'].value_counts().sort_values(ascending=True)
    class_counts.to_csv(OUTPUT_ROOT / f'class_counts_{label}.csv', header=['count'])

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 3]})
    axes[0].bar([label], [len(subset)], color='#4C78A8')
    axes[0].set_title(f'{label}: total BSD35k samples')
    axes[0].set_ylabel('count')
    axes[0].bar_label(axes[0].containers[0], fmt='%d')

    axes[1].barh(class_counts.index.astype(str), class_counts.values, color='#59A14F')
    axes[1].set_title(f'{label}: class counts')
    axes[1].set_xlabel('count')
    axes[1].tick_params(axis='y', labelsize=8)
    fig.tight_layout()
    save_path = PLOTS_DIR / f'bsd35k_{label}_counts.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', save_path)

for label, subset in v4_subsets.items():
    plot_subset_total_and_class_counts(label, subset)

## 4. Build downstream datasets

This version runs only the three requested v4-filtered BSD35k additions to keep runtime manageable.

In [ ]:
addition_sets = {
    'v4_ge2': v4_subsets['ge2'],
    'v4_ge3': v4_subsets['ge3'],
    'v4_ge4': v4_subsets['ge4'],
}

dataset_rows = []
for label, add_df in addition_sets.items():
    combined = pd.concat([train_pool, add_df[train_pool.columns]], ignore_index=True)
    dataset_path = DATASET_DIR / f'{label}_train_dataset.csv'
    combined.to_csv(dataset_path, index=False)
    add_df.to_csv(DATASET_DIR / f'{label}_added_bsd35k_rows.csv', index=False)
    dataset_rows.append({
        'dataset_label': label,
        'bsd10k_train80_samples': len(train_pool),
        'added_bsd35k_samples': len(add_df),
        'total_train_pool_samples': len(combined),
        'added_bsd35k_classes': add_df['class'].nunique() if len(add_df) else 0,
        'total_classes': combined['class'].nunique(),
        'dataset_csv': str(dataset_path),
    })

dataset_manifest = pd.DataFrame(dataset_rows)
dataset_manifest.to_csv(OUTPUT_ROOT / 'dataset_manifest.csv', index=False)
display(dataset_manifest)

## 5. Train DCASE baseline on each dataset

This uses the same model and evaluation code from `dcase2026_task1_baseline` via `confidence_baseline_common.train_and_evaluate_one`.

To reduce runtime while debugging, edit `RUN_DATASET_LABELS` before executing the cell.

In [ ]:
RUN_DATASET_LABELS = [
    'v4_ge2',
    'v4_ge3',
    'v4_ge4',
]

def make_combined_dataset(addition_df: pd.DataFrame) -> pd.DataFrame:
    if len(addition_df) == 0:
        return train_pool.reset_index(drop=True).copy()
    return pd.concat([train_pool, addition_df[train_pool.columns]], ignore_index=True).reset_index(drop=True)

def run_one_dataset(dataset_label: str, combined_df: pd.DataFrame) -> list[dict]:
    rows = []
    dataset_dir = OUTPUT_ROOT / dataset_label
    dataset_dir.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(dataset_dir / 'combined_train_pool.csv', index=False)

    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = list(splitter.split(np.zeros(len(combined_df)), combined_df['class_idx']))

    for mode in BASELINE_MODES:
        for fold, (train_idx, val_idx) in enumerate(splits):
            split_name = f'fold_{fold}'
            run_dir = dataset_dir / mode / split_name
            tr_df = combined_df.iloc[train_idx].reset_index(drop=True)
            va_df = combined_df.iloc[val_idx].reset_index(drop=True)

            base_row = {
                'dataset_label': dataset_label,
                'mode': mode,
                'split': split_name,
                'train_pool_samples_before_kfold': int(len(combined_df)),
                'train_samples': int(len(tr_df)),
                'val_samples': int(len(va_df)),
                'final_test_samples': int(len(final_test)),
                'bsd10k_train80_samples': int(len(train_pool)),
                'added_bsd35k_samples': int(max(len(combined_df) - len(train_pool), 0)),
                'output_dir': str(run_dir),
            }

            metrics_path = run_dir / 'evaluation' / 'results.txt'
            if metrics_path.exists():
                metrics = cbc.read_metrics_file(metrics_path)
                rows.append({**base_row, 'status': 'completed_existing', **metrics})
                print('[skip] existing completed run:', run_dir)
                continue

            best_val_acc, metrics = cbc.train_and_evaluate_one(
                tr_df,
                va_df,
                final_test,
                class_dict,
                top_class_dict,
                run_dir,
                mode=mode,
                seed=SEED,
                batch_size=BATCH_SIZE,
                num_epochs=NUM_EPOCHS,
                lr=0.001,
                patience=5,
                early_stopping_factor=3,
            )
            rows.append({**base_row, 'status': 'ok', 'best_val_accuracy': float(best_val_acc), **metrics})
            pd.DataFrame(rows).to_csv(dataset_dir / 'summary_results_partial.csv', index=False)
    return rows

all_rows = []
for dataset_label in RUN_DATASET_LABELS:
    print('\n' + '=' * 100)
    print('Running dataset:', dataset_label)
    combined_df = make_combined_dataset(addition_sets[dataset_label])
    rows = run_one_dataset(dataset_label, combined_df)
    all_rows.extend(rows)
    pd.DataFrame(all_rows).to_csv(OUTPUT_ROOT / 'summary_results.csv', index=False)

summary = pd.DataFrame(all_rows)
summary.to_csv(OUTPUT_ROOT / 'summary_results.csv', index=False)
display(summary)

## 6. View fold-level results

In [ ]:
summary_path = OUTPUT_ROOT / 'summary_results.csv'
summary = pd.read_csv(summary_path)
numeric_cols = [
    'best_val_accuracy', 'accuracy', 'top_accuracy', 'macro_accuracy', 'macro_top_accuracy',
    'hierarchical_accuracy', 'hierarchical_precision', 'hierarchical_recall', 'hierarchical_f1',
    'train_pool_samples_before_kfold', 'train_samples', 'val_samples', 'final_test_samples',
    'bsd10k_train80_samples', 'added_bsd35k_samples'
]
for col in numeric_cols:
    if col in summary.columns:
        summary[col] = pd.to_numeric(summary[col], errors='coerce')

view_cols = [
    'dataset_label', 'mode', 'split', 'bsd10k_train80_samples', 'added_bsd35k_samples',
    'train_pool_samples_before_kfold', 'train_samples', 'val_samples', 'final_test_samples',
    'best_val_accuracy', 'accuracy', 'top_accuracy', 'macro_accuracy', 'macro_top_accuracy',
    'hierarchical_accuracy', 'hierarchical_precision', 'hierarchical_recall', 'hierarchical_f1',
    'output_dir'
]
view = summary[[c for c in view_cols if c in summary.columns]].sort_values(
    ['dataset_label', 'mode', 'split']
).reset_index(drop=True)
view.to_csv(OUTPUT_ROOT / 'all_run_results_view.csv', index=False)
display(view)

## 7. Mean ± std summary

This prints the compact format requested, while the previous cell keeps all fold-level details.

In [ ]:
SUMMARY_METRICS = [
    'accuracy',
    'hierarchical_accuracy',
    'hierarchical_f1',
    'hierarchical_precision',
    'hierarchical_recall',
    'macro_accuracy',
    'macro_top_accuracy',
    'top_accuracy',
]

def summarize_mean_std(summary_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset_label, group in summary_df.groupby('dataset_label', sort=False):
        row = {'dataset_label': dataset_label, 'fold_count': len(group)}
        for col in ['bsd10k_train80_samples', 'added_bsd35k_samples', 'final_test_samples']:
            if col in group.columns:
                row[col] = int(group[col].iloc[0])
        for metric in SUMMARY_METRICS:
            values = pd.to_numeric(group[metric], errors='coerce').dropna()
            row[f'{metric}_mean'] = values.mean() if len(values) else np.nan
            row[f'{metric}_std'] = values.std(ddof=1) if len(values) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

fold_summary = summarize_mean_std(summary)
fold_summary.to_csv(OUTPUT_ROOT / 'fold_metric_summary_mean_std.csv', index=False)
display(fold_summary)

for _, row in fold_summary.iterrows():
    print('\n' + str(row['dataset_label']))
    print('-' * len(str(row['dataset_label'])))
    print(f"  folds                   : {int(row['fold_count'])}")
    print(f"  BSD10k train80 samples  : {int(row['bsd10k_train80_samples'])}")
    print(f"  added BSD35k samples    : {int(row['added_bsd35k_samples'])}")
    for metric in SUMMARY_METRICS:
        mean = row[f'{metric}_mean']
        std = row[f'{metric}_std']
        print(f"  {metric:<22}: {mean:.2f}% ± {std:.2f}%")

## 8. Loss curves

The baseline trainer records training loss and validation accuracy. This cell plots training loss for every fold.

In [ ]:
def load_history(run_dir: str | Path) -> dict:
    history_path = Path(run_dir) / 'history.json'
    if not history_path.exists():
        return {}
    return json.loads(history_path.read_text(encoding='utf-8'))

def plot_loss_curves(summary_df: pd.DataFrame):
    for dataset_label, group in summary_df.groupby('dataset_label', sort=False):
        fig, ax1 = plt.subplots(figsize=(12, 6))
        ax2 = ax1.twinx()
        plotted = False
        for _, row in group.iterrows():
            hist = load_history(row['output_dir'])
            if not hist:
                continue
            loss = hist.get('train_total_loss') or hist.get('train_cls_loss')
            val_acc = hist.get('val_accuracy')
            label = f"{row['mode']}/{row['split']}"
            if loss:
                ax1.plot(range(1, len(loss) + 1), loss, alpha=0.8, label=f'loss {label}')
                plotted = True
            if val_acc:
                ax2.plot(range(1, len(val_acc) + 1), val_acc, alpha=0.35, linestyle='--', label=f'val acc {label}')
        ax1.set_title(f'Training loss curves: {dataset_label}')
        ax1.set_xlabel('epoch')
        ax1.set_ylabel('train loss')
        ax2.set_ylabel('validation accuracy (%)')
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(handles1 + handles2, labels1 + labels2, fontsize=8, loc='best')
        fig.tight_layout()
        save_path = PLOTS_DIR / f'loss_curves_{dataset_label}.png'
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
        if plotted:
            plt.show()
        else:
            plt.close(fig)
        print('saved:', save_path)

plot_loss_curves(summary)

## 9. Confusion matrices

For each dataset, this cell selects the best fold by `hierarchical_accuracy` and shows its normalized confusion matrix. Each run also writes its own matrix under `output_dir/evaluation/`.

In [ ]:
def plot_confusion_from_csv(cm_path: Path, title: str, save_path: Path):
    cm_df = pd.read_csv(cm_path, index_col=0)
    labels = cm_df.index.astype(str).tolist()
    cm = cm_df.to_numpy(dtype=float)
    fig_size = max(10, len(labels) * 0.55)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues', vmin=0.0, vmax=1.0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    cbc.annotate_confusion_matrix(ax, cm, fontsize=6)
    fig.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', save_path)

best_rows = []
for dataset_label, group in summary.groupby('dataset_label', sort=False):
    best_idx = pd.to_numeric(group['hierarchical_accuracy'], errors='coerce').idxmax()
    best = summary.loc[best_idx]
    best_rows.append(best)
    cm_path = Path(best['output_dir']) / 'evaluation' / 'confusion_matrix_normalized_true.csv'
    save_path = PLOTS_DIR / f'best_confusion_matrix_{dataset_label}.png'
    title = (
        f"{dataset_label} | {best['mode']}/{best['split']} | "
        f"hierarchical_accuracy={best['hierarchical_accuracy']:.2f}%"
    )
    plot_confusion_from_csv(cm_path, title, save_path)

best_folds = pd.DataFrame(best_rows)
best_folds.to_csv(OUTPUT_ROOT / 'best_folds_by_hierarchical_accuracy.csv', index=False)
display(best_folds[[
    'dataset_label', 'mode', 'split', 'added_bsd35k_samples', 'accuracy',
    'hierarchical_accuracy', 'hierarchical_f1', 'output_dir'
]])

## 10. Quick interpretation table

Use this table to compare:

- `v4_ge2/ge3/ge4` trend: how strictness trades off data amount and downstream performance.

In [ ]:
ranked = fold_summary.sort_values('hierarchical_accuracy_mean', ascending=False).reset_index(drop=True)
display_cols = [
    'dataset_label', 'added_bsd35k_samples', 'fold_count',
    'accuracy_mean', 'accuracy_std',
    'hierarchical_accuracy_mean', 'hierarchical_accuracy_std',
    'hierarchical_f1_mean', 'hierarchical_f1_std',
    'macro_accuracy_mean', 'macro_accuracy_std',
]
ranked[display_cols].to_csv(OUTPUT_ROOT / 'ranked_compact_summary.csv', index=False)
display(ranked[display_cols])